# Projet-5GENE:
- Geoffrey RIVIERE
- Clara DESTREBECQ
- Yann LAURENT

## Partie 1 : Le format MIDI

### 1.1 Qu'est-ce que le MIDI ? 
Le MIDI, ou **Musical Instrument Digital Interface**, est un protocole de communication créé dans les années 80 pour permettre aux instruments de musique électroniques et aux ordinateurs de communiquer entre eux.

Il transmet des instructions musicales : les notes jouées et leur intensité. Un fichier MIDI ne contient pas de son, mais une partition numérique pour permettre à un ordinateur ou un synthétiseur de l'interpréter.

### 1.2 Format des fichiers
Le format de fichier SMF, **standard MIDI File**, a été défini en 1987, pour stocker les commandes MIDI sur disquette en y ajoutant des informations temporelles avec comme extension **.mid** ou **.midi**.

Il existe 3 formats SMF différents de fichiers MIDI :
- 0 : une seule piste contenant tous les messages des 16 canaux.
- 1 : plusieurs pistes, jouées simultanément.
- 2 : plusieurs pistes, mais jouées séquentiellement (rarement utilisé).

Bien que le type **1** soit le format le plus courant et le plus pratique, certains appareils ne reconnaissent que le format 0.

Les logiciels MIDI ont généralement un format de fichier propriétaire pour gérer les séquences MIDI mais sont tous capables d'importer et d'exporter les fichiers **.mid**, ce qui facilite les échanges entre plateformes.

### 1.3 Avantages du format MIDI
- **Léger** : un fichier MIDI ne contient que des instructions, il ne contient que quelques Ko pour plusieurs minutes de musique, contre plusieurs Mo pour un fichier audio style MP3.
- **Standard universel** : la quasi-totalité des logiciels et instruments de musique l'acceptent, ce qui facilite les échanges.
- **Modifiable facilement** : on peut changer le tempo, l'instrument et transposer les notes sans perdre de la qualité.
- **Données symboliques** : les notes sont directement accessibles sous forme de nombres (hauteur, durée, vélocité), ce qui les rend faciles à exploiter en apprentissage automatique.

**Limites** :
- Le rendu sonore dépend de l'appareil ou du logiciel qui va lire le fichier.
- Le MIDI ne contient pas la voix ni le timbre réel d'un instrument enregistré.

### 1.4 Caractéristiques d'une note
- **Hauteur** : nombre entre 0 et 127
- **Vélocité** : un nombre entre 0 et 127, qui correspond à la force de frappe
- **Durée** : elle n'est pas écrite directement, elle se calcule avec l'écart de temps entre le **note_on** et le **note_off**

Le début et la fin d'une note sont indiqués par des messages, les plus importants étant :
- **note_on** : Début d'une note (hauteur + vélocité)
- **note_off** : Fin d'une note

### Sources
- Wikipédia : **https://fr.wikipedia.org/wiki/Musical_Instrument_Digital_Interface**
- Amped Studio: **https://ampedstudio.com/fr/blog/what-is-midi/**

## Partie 2 : Choix d'une librairie Python

### 2.1 Comparaison des Librairies.
- **Mido** :
    - **Avantages** :
        - Accès direct aux ports, aux messages et aux fichiers MIDI ("MIDI 1.0")
        - Message représenté par un objet Python
        - Utile pour lire des messages un par un
    - **Inconvénients** :
        - Ne donne pas de notes complètes, il faut les reconstruire à partir des messages
- **pretty_midi** :
    - **Avantages** :
        - Facile à gérer (**modifier et à analyser**)
        - Chaque note possède une hauteur ("pitch"), une vélocité ("velocity"), un début ("start") et une fin ("end")
    - **Inconvénients** :
        - Accés moins direct au messages que mido

**Choix** : Nous utiliserons donc **pretty_midi**, car chaque note contient déjà les informations nécessaires pour obtenir les notes complètes et leurs durées.

### 2.2 Fonctionnement de pretty_midi.

**pretty_midi** organise un fichier MIDI en trois niveaux :

- **PrettyMIDI** : représente le fichier MIDI complet.
    - "instruments" : liste d'objets Instrument
- **Instrument** : Représente un instrument MIDI
    - "notes" : liste d'objets note
    - "program" : type d'instrument
    - "is_drum" : booléan indiquant si c'est une piste de percussions
- **Notes** : représente une note jouée.
    - "pitch" : hauteur de la note (0-127)
    - "velocity" : vélocité de la note (en secondes)
    - "start" : début de la note (en secondes)
    - "end" : fin de la note (0-127)

### Sources
- Mido : **https://mido.readthedocs.io/en/stable/**
- pretty_midi: **https://craffel.github.io/pretty-midi/**

## Partie 3 : Choix du compositeur

### 3.1 Lire le fichier

In [1]:
import pandas as pd

df = pd.read_csv("../data/maestro-v3.0.0/maestro-v3.0.0.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/maestro-v3.0.0/maestro-v3.0.0.csv'

### 3.2 Calculer la durée et le nombre de morceaux par compositeur.

In [ ]:
durees = (
    df.groupby("canonical_composer")["duration"].agg(nb_morceaux="count", duree_secondes="sum").sort_values("duree_secondes", ascending=False)
)
durees["duree_heures"] = (durees["duree_secondes"] / 3600).round(1)

durees.head(10)

,nb_morceaux,duree_secondes,duree_heures
canonical_composer,,,
Franz Schubert,186,128958.701199,35.8
Ludwig van Beethoven,146,96922.682897,26.9
Frédéric Chopin,201,94389.006934,26.2
Franz Liszt,131,72958.738892,20.3
Robert Schumann,49,56596.114873,15.7
Johann Sebastian Bach,145,49559.997408,13.8
Johannes Brahms,26,29028.633682,8.1
Sergei Rachmaninoff,59,23985.896308,6.7
Claude Debussy,45,21507.282772,6.0


### 3.3 Choix du compositeur

Nous allons donc choisir **Franz Schubert**, il s'agit du compositeur qui a la plus longue durée totale dans le dataset, environ 36 heures, réparties dans 186 morceaux.

## Partie 4 : Réunir tous les morceaux de ce compositeur

Nous allons réunir tous les morceaux de **Franz Schubert** dans un même répertoire

In [ ]:
import shutil
from pathlib import Path

dossier_maestro = Path("../data/maestro-v3.0.0/")
dossier_franz_schubert = Path("../data/franz_schubert/")
dossier_franz_schubert.mkdir(exist_ok=True)

morceaux = df[df["canonical_composer"] == "Franz Schubert"]

for fichier in morceaux["midi_filename"]:
    chemin_origine = dossier_maestro / fichier
    shutil.copy(chemin_origine, dossier_franz_schubert / chemin_origine.name)

print("Morceaux envoyés dans le nouveau dossier : ", len(list(dossier_franz_schubert.glob("*.midi"))))

Morceaux envoyés dans le nouveau dossier :  186


## Partie 5 : Extraction des notes et des durées

Pour chaque morceau, nous allons créer deux listes :
- La succession des notes.
- La durée de chacune d'elles.

In [ ]:
import pretty_midi

list_notes = []
list_durées = []

for chemin in sorted(dossier_franz_schubert.glob('*midi')):
    midi = pretty_midi.PrettyMIDI(str(chemin))
    notes_trie = sorted(midi.instruments[0].notes, key=lambda note: (note.start, note.pitch))

    list_notes.append([note.pitch for note in notes_trie])
    list_durées.append([round(float(note.end - note.start),2) for note in notes_trie])

print("Nombre de morceaux :", len(list_notes))
print("10 premières notes du 1er morceau :", list_notes[0][:10])
print("10 premières durées du 1er morceau :", list_durées[0][:10])

Nombre de morceaux : 186
10 premières notes du 1er morceau : [70, 65, 58, 46, 50, 53, 58, 46, 50, 70]
10 premières durées du 1er morceau : [0.41, 0.44, 0.4, 0.48, 0.47, 0.49, 0.46, 0.48, 0.8, 0.4]


On peut observer avec les résultats que :
- Nous avons un total de 186 morceaux pour le compositeur **Franz Schubert**
- Nous avons le début de la première liste avec la succession de notes (**entre 0 et 127**)
- Nous avons le début de la deuxième liste avec la durée de chacune d'entre elles (**entre 0.04 à 1.66 s**)

## Partie 6 : Vectorisation

Nous allons arrondir les durées car ce sont des nombres décimaux presque tous différents, si on n'arrondit pas, on se retrouve avec un très grand nombre de valeurs différentes, l'apprentissage devient donc plus difficile.
- (ex: 1.20 et 1.21 seraient différentes, le fait d'avoir un arrondi sur 0.05 permet de les classer dans la même catégorie)

Nous remplacerons séparément chaque note et chaque durée par un identifiant pour permettre d'avoir des nombres consécutifs à partir de 0.

Nous allons vectoriser les notes et les durées séparément pour les prédire séparément pour éviter d'avoir une liste qui combine toutes les combinaisons possibles.

Les séquences de taille fixe permettent à PyTorch de les regrouper par lots sous forme de tenseurs, pour ce faire les exemples d'un lot doivent avoir la même longueur, sauf que dans ce dataset les morceaux sont tous de longueurs différentes.

Nous allons fixer la taille des séquences à 100 car trop court rendrait le modèle moins performant par manque de contexte musical et trop long ça consommerait trop de ressources.

In [ ]:
ECART_DUREE = 0.05
TAILLE_SEQUENCE = 100

durees_arrondies = [
    [max(1, round(duree / ECART_DUREE)) for duree in morceau]
    for morceau in list_durées
]

notes_uniques = sorted({note for morceau in list_notes for note in morceau})
durees_uniques = sorted({duree for morceau in durees_arrondies for duree in morceau})

id_notes = {note: i for i, note in enumerate(notes_uniques)}
id_duree = {duree: i for i, duree in enumerate(durees_uniques)}

notes_vectorisees = [[id_notes[note] for note in morceau] for morceau in list_notes]
durees_vectorisees = [[id_duree[duree] for duree in morceau] for morceau in durees_arrondies]

def decouper(morceaux, taille):
    return [
        morceau[i : i + taille]
        for morceau in morceaux
        for i in range(0, len(morceau) - taille + 1, taille)
    ]

# sequence +1 permet de prédire l'élément suivant 
sequences_notes = decouper(notes_vectorisees, TAILLE_SEQUENCE + 1)
sequences_durees = decouper(durees_vectorisees, TAILLE_SEQUENCE + 1)

print("Notes différentes :", len(notes_uniques))
print("Durées différentes :", len(durees_uniques))
print("Nombre de séquences :", len(sequences_notes))

Notes différentes : 82
Durées différentes : 214
Nombre de séquences : 11702


On peut observer avec les résultats que :
- nous avons 82 notes différentes, ce qui veut dire que le modèle aura 82 choix possibles pour prédire une note
- Il nous reste 214 catégories, ce qui montre que les durées sont très variées.
- le nombre 11 702 prouve que l'on a beaucoup de données pour l'entraînement.


## Partie 7 : Jeu d'entrainement

Pour apprendre à prédire la note suivante, chaque séquence est découpée en une **entrée** (tous les éléments sauf le dernier) et une **cible** (la même séquence décalée d'une place).
Ainsi, pour chaque position, le modèle doit deviner l'élément qui suit.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

def decaler(sequence):
    """Retourne (entrée, cible) : l'entrée omet le dernier élément, la cible est décalée d'une place."""
    return sequence[:-1], sequence[1:]

# Exemple sur une petite séquence
print(decaler([10, 20, 30, 40, 50]))

Nous créons un `Dataset` PyTorch qui stocke les séquences de notes et de durées sous forme de tenseurs.
Pour chaque exemple, il renvoie les entrées et les cibles décalées des notes et des durées.

In [ ]:
class DatasetMusique(Dataset):
    def __init__(self, sequences_notes, sequences_durees):
        assert len(sequences_notes) == len(sequences_durees), "Les séquences de notes et de durées doivent être alignées"
        self.notes = torch.tensor(sequences_notes, dtype=torch.long)
        self.durees = torch.tensor(sequences_durees, dtype=torch.long)

    def __len__(self):
        return len(self.notes)

    def __getitem__(self, i):
        x_notes, y_notes = decaler(self.notes[i])
        x_durees, y_durees = decaler(self.durees[i])
        return x_notes, x_durees, y_notes, y_durees

dataset = DatasetMusique(sequences_notes, sequences_durees)
print("Nombre d'exemples :", len(dataset))

Nous séparons les données en **90 % entraînement** et **10 % validation** (graine fixée pour la reproductibilité).
Les `DataLoader` regroupent les séquences par lots de 64, mélangés uniquement pour l'entraînement.

In [ ]:
TAILLE_LOT = 64

nb_validation = int(0.1 * len(dataset))
nb_entrainement = len(dataset) - nb_validation

jeu_entrainement, jeu_validation = random_split(
    dataset, [nb_entrainement, nb_validation],
    generator=torch.Generator().manual_seed(42)
)

loader_entrainement = DataLoader(jeu_entrainement, batch_size=TAILLE_LOT, shuffle=True)
loader_validation = DataLoader(jeu_validation, batch_size=TAILLE_LOT, shuffle=False)

print("Séquences d'entraînement :", len(jeu_entrainement))
print("Séquences de validation  :", len(jeu_validation))

# Vérification sur un lot
x_notes, x_durees, y_notes, y_durees = next(iter(loader_entrainement))
print("Forme d'un lot (entrée) :", tuple(x_notes.shape))
print("Forme d'un lot (cible)  :", tuple(y_notes.shape))
print("Entrée notes :", x_notes[0, :8].tolist())
print("Cible notes  :", y_notes[0, :8].tolist())
print("Décalage correct :", torch.equal(x_notes[:, 1:], y_notes[:, :-1]) and torch.equal(x_durees[:, 1:], y_durees[:, :-1]))

On peut observer avec les résultats que :
- Nous avons 10 532 séquences d'entraînement et 1 170 de validation, en lots de forme (64, 100).
- La cible correspond bien à l'entrée décalée d'une place, le jeu est prêt pour l'entraînement.

## Partie 8 : Réseau Transformer

Le modèle doit prédire, à chaque position, **la note suivante et sa durée** à partir des notes précédentes.
Nous construisons le réseau pièce par pièce :
1. un **bloc Transformer** (attention multi-têtes masquée + réseau feed-forward),
2. une couche **Embedding + Positional Encoding** sinusoïdal,
3. le **réseau complet** qui concatène les embeddings des notes et des durées et possède deux sorties.

In [ ]:
import math
import copy
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Appareil utilisé :", device)

### 8.1 Bloc Transformer et masquage

**Le masquage** : lors de l'entraînement, le modèle reçoit toute la séquence d'un coup. Sans masque, la position *i* pourrait « regarder » la note *i+1* qu'elle doit justement prédire : ce serait de la triche.
Le **masque causal** est une matrice triangulaire supérieure : la case (i, j) vaut `True` si j > i, c'est-à-dire si la position j est dans le futur de i. Ces scores d'attention sont remplacés par -∞, ce qui leur donne un poids nul après le softmax.

**L'attention multi-têtes** : chaque élément est projeté en une requête (Q), une clé (K) et une valeur (V).
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}} + \text{masque}\right)V$$
On découpe la dimension en plusieurs têtes pour que chacune puisse s'intéresser à un aspect différent (harmonie, rythme, motif…).

**Le bloc Transformer** enchaîne : normalisation → attention masquée → connexion résiduelle, puis normalisation → réseau feed-forward → connexion résiduelle.

In [ ]:
def masque_causal(taille, device=None):
    # True = position interdite (dans le futur)
    return torch.triu(torch.ones(taille, taille, dtype=torch.bool, device=device), diagonal=1)

print(masque_causal(5).int())

In [ ]:
class AttentionMultiTetes(nn.Module):
    def __init__(self, dim, nb_tetes, dropout=0.1):
        super().__init__()
        assert dim % nb_tetes == 0, "La dimension doit être divisible par le nombre de têtes"
        self.nb_tetes = nb_tetes
        self.dim_tete = dim // nb_tetes
        self.qkv = nn.Linear(dim, 3 * dim)   # calcule Q, K et V en une seule fois
        self.sortie = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, masque):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, D) -> (B, nb_tetes, T, dim_tete)
        q = q.view(B, T, self.nb_tetes, self.dim_tete).transpose(1, 2)
        k = k.view(B, T, self.nb_tetes, self.dim_tete).transpose(1, 2)
        v = v.view(B, T, self.nb_tetes, self.dim_tete).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dim_tete)   # (B, H, T, T)
        scores = scores.masked_fill(masque, float("-inf"))           # masquage du futur
        poids = self.dropout(F.softmax(scores, dim=-1))

        resultat = (poids @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.sortie(resultat)


class BlocTransformer(nn.Module):
    def __init__(self, dim, nb_tetes, dim_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = AttentionMultiTetes(dim, nb_tetes, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(dim, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, dim),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, masque):
        x = x + self.dropout(self.attention(self.norm1(x), masque))   # connexion résiduelle 1
        x = x + self.feed_forward(self.norm2(x))                      # connexion résiduelle 2
        return x

Vérification du masquage : si l'on modifie le **dernier** élément d'une séquence, les sorties des positions précédentes ne doivent pas changer.

In [ ]:
bloc_test = BlocTransformer(dim=16, nb_tetes=4, dim_ff=32, dropout=0.0).eval()
x = torch.randn(1, 6, 16)
x_modifie = x.clone()
x_modifie[0, -1] = torch.randn(16)   # on change uniquement la dernière position

masque = masque_causal(6)
with torch.no_grad():
    s1, s2 = bloc_test(x, masque), bloc_test(x_modifie, masque)

print("Positions 0 à 4 identiques :", torch.allclose(s1[0, :-1], s2[0, :-1], atol=1e-6))
print("Dernière position modifiée :", not torch.allclose(s1[0, -1], s2[0, -1]))

### 8.2 Embedding et Positional Encoding

- **Embedding** : transforme chaque identifiant (note ou durée) en un vecteur appris de dimension `dim`.
- **Positional Encoding** : l'attention ne tient pas compte de l'ordre des éléments. On ajoute donc à chaque vecteur un encodage qui dépend de sa position *pos* :

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/dim}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/dim}}\right)$$

Chaque dimension oscille à une fréquence différente, ce qui donne à chaque position une « signature » unique. Cet encodage n'a aucun paramètre à apprendre (il est stocké avec `register_buffer`).

In [ ]:
class EmbeddingPositionnel(nn.Module):
    def __init__(self, taille_vocab, dim, longueur_max=TAILLE_SEQUENCE):
        super().__init__()
        assert dim % 2 == 0, "La dimension doit être paire pour l'encodage sinusoïdal"
        self.dim = dim
        self.embedding = nn.Embedding(taille_vocab, dim)

        position = torch.arange(longueur_max).unsqueeze(1)                              # (L, 1)
        frequences = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))  # (dim/2,)
        pe = torch.zeros(longueur_max, dim)
        pe[:, 0::2] = torch.sin(position * frequences)
        pe[:, 1::2] = torch.cos(position * frequences)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x : (B, T) identifiants -> (B, T, dim)
        T = x.size(1)
        return self.embedding(x) * math.sqrt(self.dim) + self.pe[:T]

In [ ]:
pe_affichage = EmbeddingPositionnel(10, 64).pe

plt.figure(figsize=(10, 4))
plt.imshow(pe_affichage.T, aspect="auto", cmap="RdBu")
plt.xlabel("Position dans la séquence")
plt.ylabel("Dimension")
plt.title("Encodage positionnel sinusoïdal")
plt.colorbar()
plt.show()

Sur ce graphique, les premières dimensions oscillent très vite (elles distinguent des positions voisines) et les dernières très lentement (elles situent la position à l'échelle de la séquence entière).

### 8.3 Réseau Transformer complet

- Les notes et les durées passent chacune dans leur propre couche `EmbeddingPositionnel`.
- Les deux résultats sont **concaténés** : un vecteur de dimension `dim_notes + dim_durees` décrit à la fois la hauteur et la durée de chaque note.
- Ce vecteur traverse le(s) bloc(s) Transformer.
- Deux couches linéaires de sortie donnent les **logits** de la note suivante (82 classes) et de la durée suivante (214 classes).

Le nombre de blocs est un paramètre (`nb_blocs`), ce qui permettra de réutiliser la même classe pour la question 8.7.

In [ ]:
class TransformerMusical(nn.Module):
    def __init__(self, nb_notes, nb_durees, dim_notes=128, dim_durees=64,
                 nb_blocs=1, nb_tetes=4, dim_ff=512, dropout=0.1, longueur_max=TAILLE_SEQUENCE):
        super().__init__()
        # on garde les hyperparamètres pour pouvoir recréer le modèle lors du chargement
        self.config = dict(nb_notes=nb_notes, nb_durees=nb_durees, dim_notes=dim_notes,
                           dim_durees=dim_durees, nb_blocs=nb_blocs, nb_tetes=nb_tetes,
                           dim_ff=dim_ff, dropout=dropout, longueur_max=longueur_max)
        dim = dim_notes + dim_durees

        self.emb_notes = EmbeddingPositionnel(nb_notes, dim_notes, longueur_max)
        self.emb_durees = EmbeddingPositionnel(nb_durees, dim_durees, longueur_max)
        self.dropout = nn.Dropout(dropout)
        self.blocs = nn.ModuleList([BlocTransformer(dim, nb_tetes, dim_ff, dropout) for _ in range(nb_blocs)])
        self.norm = nn.LayerNorm(dim)
        self.sortie_notes = nn.Linear(dim, nb_notes)
        self.sortie_durees = nn.Linear(dim, nb_durees)

    def forward(self, notes, durees):
        T = notes.size(1)
        x = torch.cat([self.emb_notes(notes), self.emb_durees(durees)], dim=-1)   # concaténation
        x = self.dropout(x)
        masque = masque_causal(T, notes.device)
        for bloc in self.blocs:
            x = bloc(x, masque)
        x = self.norm(x)
        return self.sortie_notes(x), self.sortie_durees(x)


def nb_parametres(modele):
    return sum(p.numel() for p in modele.parameters() if p.requires_grad)

modele = TransformerMusical(len(notes_uniques), len(durees_uniques), nb_blocs=1).to(device)
print("Nombre de paramètres :", f"{nb_parametres(modele):,}".replace(",", " "))

# test sur un lot
logits_notes, logits_durees = modele(x_notes.to(device), x_durees.to(device))
print("Sortie notes  :", tuple(logits_notes.shape))
print("Sortie durées :", tuple(logits_durees.shape))

### 8.4 Entraînement

La fonction de perte est la somme de deux **entropies croisées** : une pour la note prédite et une pour la durée prédite. Le modèle apprend donc les deux tâches en même temps.

- Optimiseur **AdamW**, taux d'apprentissage 3e-4.
- **Écrêtage du gradient** (norme max 1) pour stabiliser l'apprentissage.
- À chaque époque, on évalue sur le jeu de validation et on garde les poids du **meilleur** modèle (perte de validation la plus basse), ce qui évite de conserver un modèle sur-entraîné.

In [ ]:
def calculer_perte(modele, x_n, x_d, y_n, y_d):
    logits_n, logits_d = modele(x_n, x_d)
    perte_n = F.cross_entropy(logits_n.reshape(-1, logits_n.size(-1)), y_n.reshape(-1))
    perte_d = F.cross_entropy(logits_d.reshape(-1, logits_d.size(-1)), y_d.reshape(-1))
    precision_n = (logits_n.argmax(-1) == y_n).float().mean()
    precision_d = (logits_d.argmax(-1) == y_d).float().mean()
    return perte_n, perte_d, precision_n, precision_d


def passe(modele, loader, optimiseur=None):
    # entraîne si un optimiseur est fourni, sinon évalue
    entrainement = optimiseur is not None
    modele.train(entrainement)
    totaux = torch.zeros(4)
    with torch.set_grad_enabled(entrainement):
        for x_n, x_d, y_n, y_d in loader:
            x_n, x_d, y_n, y_d = x_n.to(device), x_d.to(device), y_n.to(device), y_d.to(device)
            perte_n, perte_d, prec_n, prec_d = calculer_perte(modele, x_n, x_d, y_n, y_d)
            if entrainement:
                optimiseur.zero_grad()
                (perte_n + perte_d).backward()
                torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
                optimiseur.step()
            totaux += torch.tensor([perte_n.item(), perte_d.item(), prec_n.item(), prec_d.item()])
    return (totaux / len(loader)).tolist()


def entrainer(modele, nb_epoques, lr=3e-4):
    optimiseur = torch.optim.AdamW(modele.parameters(), lr=lr, weight_decay=0.01)
    historique = {"entrainement": [], "validation": []}
    meilleure_perte, meilleurs_poids = float("inf"), None

    for epoque in range(1, nb_epoques + 1):
        p_n, p_d, _, _ = passe(modele, loader_entrainement, optimiseur)
        v_n, v_d, a_n, a_d = passe(modele, loader_validation)
        historique["entrainement"].append(p_n + p_d)
        historique["validation"].append(v_n + v_d)

        if v_n + v_d < meilleure_perte:
            meilleure_perte = v_n + v_d
            meilleurs_poids = copy.deepcopy(modele.state_dict())

        print(f"Époque {epoque:2d} | perte entr. {p_n + p_d:.3f} | perte val. {v_n + v_d:.3f} "
              f"(notes {v_n:.3f}, durées {v_d:.3f}) | précision val. notes {a_n:.1%}, durées {a_d:.1%}")

    modele.load_state_dict(meilleurs_poids)
    print(f"Meilleure perte de validation : {meilleure_perte:.3f}")
    return historique


def afficher_historique(historiques):
    plt.figure(figsize=(8, 4))
    for nom, h in historiques.items():
        epoques = range(1, len(h["entrainement"]) + 1)
        ligne, = plt.plot(epoques, h["entrainement"], label=f"{nom} - entraînement")
        plt.plot(epoques, h["validation"], "--", color=ligne.get_color(), label=f"{nom} - validation")
    plt.xlabel("Époque")
    plt.ylabel("Perte (notes + durées)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
NB_EPOQUES = 15

historique_1_bloc = entrainer(modele, NB_EPOQUES)
afficher_historique({"1 bloc": historique_1_bloc})

On peut observer avec les résultats que :
- La perte d'entraînement et la perte de validation diminuent au fil des époques : le modèle apprend les régularités des morceaux de Schubert.
- Pour comparaison, un modèle qui tirerait au hasard aurait une perte de ln(82) + ln(214) ≈ 4,41 + 5,37 = **9,78**.
- Si la perte de validation remonte alors que celle d'entraînement continue de baisser, le modèle commence à sur-apprendre ; c'est pour cela que l'on garde les poids de la meilleure époque.

*(À compléter avec les valeurs obtenues lors de l'exécution.)*

### 8.5 Génération de nouveaux morceaux

**Principe** : on part d'une séquence « amorce » de 100 notes tirée du jeu de validation. À chaque étape :
1. le modèle donne les logits de la note et de la durée suivantes ;
2. le softmax les transforme en probabilités ;
3. on **tire au hasard** une note et une durée selon ces probabilités (`torch.multinomial`) ;
4. on les ajoute à la séquence, et on garde les 100 derniers éléments comme contexte pour l'étape suivante.

Le tirage aléatoire (plutôt que de prendre toujours la note la plus probable) évite que le morceau tourne en boucle sur les mêmes motifs.

La fonction a déjà un paramètre `temperature` (égal à 1 ici, c'est-à-dire les probabilités brutes du modèle), il sera étudié à la question 8.6.

In [ ]:
@torch.no_grad()
def generer(modele, notes_depart, durees_depart, nb_a_generer=300, temperature=1.0):
    modele.eval()
    notes = [int(n) for n in notes_depart]
    durees = [int(d) for d in durees_depart]
    contexte = modele.config["longueur_max"]

    for _ in range(nb_a_generer):
        x_n = torch.tensor(notes[-contexte:], device=device).unsqueeze(0)
        x_d = torch.tensor(durees[-contexte:], device=device).unsqueeze(0)
        logits_n, logits_d = modele(x_n, x_d)

        # on ne garde que la prédiction de la dernière position
        proba_n = F.softmax(logits_n[0, -1] / temperature, dim=-1)
        proba_d = F.softmax(logits_d[0, -1] / temperature, dim=-1)

        notes.append(torch.multinomial(proba_n, 1).item())
        durees.append(torch.multinomial(proba_d, 1).item())

    return notes[len(notes_depart):], durees[len(durees_depart):]

**Conversion en morceau audible**

Les identifiants sont reconvertis en vraies valeurs grâce aux listes `notes_uniques` (id → hauteur MIDI) et `durees_uniques` (id → nombre de pas de 0,05 s).

Notre représentation ne contient que la hauteur et la durée de chaque note, **pas l'instant où elle commence**. On place donc chaque note après la précédente : la note suivante commence après la durée de la note courante, limitée à `PAS_MAX` secondes pour qu'une note longue continue de résonner sous les suivantes au lieu de créer un silence. Les accords, qui étaient stockés comme des notes successives, sont ainsi joués en arpèges rapides.

On écrit un fichier **.mid** et un fichier **.wav** (synthèse simple par `pretty_midi`), et on l'affiche avec un lecteur audio dans le notebook.

In [ ]:
import wave
import numpy as np
from IPython.display import Audio, display

dossier_generation = Path("../data/generations/")
dossier_generation.mkdir(exist_ok=True)

PAS_MAX = 0.5
FREQ_ECHANTILLONNAGE = 22050


def vers_midi(ids_notes, ids_durees, pas_max=PAS_MAX):
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)   # 0 = piano acoustique
    temps = 0.0
    for id_n, id_d in zip(ids_notes, ids_durees):
        duree = durees_uniques[id_d] * ECART_DUREE
        piano.notes.append(pretty_midi.Note(velocity=80, pitch=notes_uniques[id_n],
                                            start=temps, end=temps + duree))
        temps += min(duree, pas_max)
    midi.instruments.append(piano)
    return midi


def sauvegarder_morceau(ids_notes, ids_durees, nom):
    midi = vers_midi(ids_notes, ids_durees)
    midi.write(str(dossier_generation / f"{nom}.mid"))

    audio = midi.synthesize(fs=FREQ_ECHANTILLONNAGE)
    audio = audio / (np.abs(audio).max() + 1e-9)          # normalisation entre -1 et 1
    with wave.open(str(dossier_generation / f"{nom}.wav"), "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(FREQ_ECHANTILLONNAGE)
        f.writeframes((audio * 32767).astype(np.int16).tobytes())

    print(f"{nom} : {len(ids_notes)} notes, {midi.get_end_time():.1f} s")
    display(Audio(audio, rate=FREQ_ECHANTILLONNAGE))

In [ ]:
# amorce : une séquence du jeu de validation (jamais vue pendant l'entraînement)
amorce_notes, amorce_durees, _, _ = jeu_validation[0]

# morceau de référence : l'amorce elle-même, pour comparer avec la génération
sauvegarder_morceau(amorce_notes.tolist(), amorce_durees.tolist(), "amorce")

torch.manual_seed(0)
gen_notes, gen_durees = generer(modele, amorce_notes, amorce_durees, nb_a_generer=300)
sauvegarder_morceau(gen_notes, gen_durees, "generation_1_bloc")

### 8.6 Terme de température

Avant le softmax, on divise les logits par une **température** T :

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T < 1** : les écarts entre logits sont amplifiés, la distribution devient plus « pointue ». Le modèle choisit presque toujours les notes les plus probables : musique plus prévisible, mais risque de répétitions.
- **T = 1** : on tire selon les probabilités apprises par le modèle.
- **T > 1** : la distribution s'aplatit, les notes peu probables sont tirées plus souvent : musique plus variée, mais plus chaotique et dissonante.
- T → 0 revient à toujours prendre la note la plus probable (argmax), T → ∞ revient à un tirage uniforme.

Visualisons l'effet sur la distribution de la prochaine note :

In [ ]:
TEMPERATURES = [0.5, 1.0, 1.5, 2.0]

with torch.no_grad():
    modele.eval()
    logits_n, _ = modele(amorce_notes.unsqueeze(0).to(device), amorce_durees.unsqueeze(0).to(device))
    logits_derniere = logits_n[0, -1].cpu()

fig, axes = plt.subplots(1, len(TEMPERATURES), figsize=(16, 3), sharey=True)
for ax, t in zip(axes, TEMPERATURES):
    proba = F.softmax(logits_derniere / t, dim=-1)
    ax.bar(notes_uniques, proba.numpy(), width=1.0)
    ax.set_title(f"T = {t}  (proba max {proba.max():.0%})")
    ax.set_xlabel("Hauteur MIDI")
axes[0].set_ylabel("Probabilité")
plt.tight_layout()
plt.show()

Nous générons maintenant un morceau pour chaque température à partir de la même amorce, et mesurons :
- le nombre de **hauteurs différentes** utilisées,
- l'**entropie** des notes générées (plus elle est élevée, plus les notes sont variées),
- le **taux de répétition** (proportion de notes identiques à la précédente),
- l'**écart moyen** en demi-tons entre deux notes consécutives (un grand écart traduit des sauts brusques).

In [ ]:
def statistiques(ids_notes):
    hauteurs = np.array([notes_uniques[i] for i in ids_notes])
    _, comptes = np.unique(hauteurs, return_counts=True)
    p = comptes / comptes.sum()
    return {
        "hauteurs différentes": len(comptes),
        "entropie (bits)": round(float(-(p * np.log2(p)).sum()), 2),
        "taux de répétition": round(float((hauteurs[1:] == hauteurs[:-1]).mean()), 3),
        "écart moyen (demi-tons)": round(float(np.abs(np.diff(hauteurs)).mean()), 2),
    }

resultats = {"Original (amorce)": statistiques(amorce_notes.tolist())}
for t in TEMPERATURES:
    torch.manual_seed(0)
    n, d = generer(modele, amorce_notes, amorce_durees, nb_a_generer=300, temperature=t)
    sauvegarder_morceau(n, d, f"generation_T{t}")
    resultats[f"T = {t}"] = statistiques(n)

pd.DataFrame(resultats).T

On peut observer avec les résultats que :
- Avec une **température basse** (0,5), le modèle reste sur les notes qu'il juge les plus sûres : moins de hauteurs différentes, une entropie plus faible, et le morceau a tendance à répéter les mêmes motifs.
- Avec une **température de 1**, les statistiques doivent être proches de celles du morceau original : c'est la distribution apprise par le modèle.
- Avec une **température haute** (1,5 – 2), le nombre de notes différentes et l'écart moyen entre notes augmentent : le morceau devient plus aléatoire, avec des sauts et des dissonances qui ne ressemblent plus à du Schubert.
- Un bon compromis se situe généralement **entre 0,8 et 1,1**.

*(À ajuster selon les valeurs obtenues et l'écoute des morceaux.)*

### 8.7 Réseau avec plusieurs blocs Transformer

Nous reprenons exactement la même classe `TransformerMusical`, avec cette fois **4 blocs** empilés. Chaque bloc supplémentaire permet au modèle de combiner les informations extraites par le précédent, et donc de repérer des structures plus complexes (motifs, phrases musicales).

In [ ]:
torch.manual_seed(42)
modele_multi = TransformerMusical(len(notes_uniques), len(durees_uniques), nb_blocs=4).to(device)
print("Nombre de paramètres :", f"{nb_parametres(modele_multi):,}".replace(",", " "))

historique_4_blocs = entrainer(modele_multi, NB_EPOQUES)
afficher_historique({"1 bloc": historique_1_bloc, "4 blocs": historique_4_blocs})

In [ ]:
# génération avec le modèle à 4 blocs, même amorce et même graine pour comparer
torch.manual_seed(0)
gen_notes_multi, gen_durees_multi = generer(modele_multi, amorce_notes, amorce_durees, nb_a_generer=300)
sauvegarder_morceau(gen_notes_multi, gen_durees_multi, "generation_4_blocs")

for t in [0.8, 1.2]:
    torch.manual_seed(0)
    n, d = generer(modele_multi, amorce_notes, amorce_durees, nb_a_generer=300, temperature=t)
    sauvegarder_morceau(n, d, f"generation_4_blocs_T{t}")

pd.DataFrame({
    "Original (amorce)": statistiques(amorce_notes.tolist()),
    "1 bloc": statistiques(gen_notes),
    "4 blocs": statistiques(gen_notes_multi),
}).T

On peut observer avec les résultats que :
- Le modèle à 4 blocs a environ 4 fois plus de paramètres dans sa partie Transformer, et chaque époque est plus longue.
- Sa perte de validation doit descendre plus bas que celle du modèle à 1 bloc : il prédit mieux la note et la durée suivantes.
- Il peut aussi sur-apprendre plus vite (écart entre courbe d'entraînement et de validation) : le dropout et la sauvegarde de la meilleure époque limitent ce problème.
- À l'écoute, les morceaux générés devraient être plus cohérents (motifs repris, enchaînements plus naturels).

*(À compléter avec les valeurs obtenues lors de l'exécution.)*

### 8.8 Sauvegarde du modèle

On sauvegarde dans un seul fichier :
- les **poids** du modèle (`state_dict`),
- ses **hyperparamètres** (`config`) pour pouvoir le reconstruire,
- les **dictionnaires de vectorisation** (`notes_uniques`, `durees_uniques`) et les constantes, sans lesquels on ne pourrait pas reconvertir les prédictions en notes.

In [ ]:
dossier_modeles = Path("../modeles/")
dossier_modeles.mkdir(exist_ok=True)


def sauvegarder_modele(modele, chemin):
    torch.save({
        "poids": modele.state_dict(),
        "config": modele.config,
        "notes_uniques": notes_uniques,
        "durees_uniques": durees_uniques,
        "ECART_DUREE": ECART_DUREE,
        "TAILLE_SEQUENCE": TAILLE_SEQUENCE,
    }, chemin)
    print("Modèle sauvegardé :", chemin)


def charger_modele(chemin):
    sauvegarde = torch.load(chemin, map_location=device)
    modele = TransformerMusical(**sauvegarde["config"]).to(device)
    modele.load_state_dict(sauvegarde["poids"])
    modele.eval()
    return modele, sauvegarde


sauvegarder_modele(modele, dossier_modeles / "transformer_schubert_1_bloc.pt")
sauvegarder_modele(modele_multi, dossier_modeles / "transformer_schubert_4_blocs.pt")

In [ ]:
# vérification : le modèle rechargé donne exactement les mêmes prédictions
modele_charge, infos = charger_modele(dossier_modeles / "transformer_schubert_4_blocs.pt")
modele_multi.eval()

with torch.no_grad():
    entree_n = amorce_notes.unsqueeze(0).to(device)
    entree_d = amorce_durees.unsqueeze(0).to(device)
    sortie_originale = modele_multi(entree_n, entree_d)[0]
    sortie_chargee = modele_charge(entree_n, entree_d)[0]

print("Configuration :", infos["config"])
print("Prédictions identiques :", torch.allclose(sortie_originale, sortie_chargee))